In [1]:
import torch
from manify.curvature_estimation.delta_hyperbolicity import delta_hyperbolicity
import networkx as nx 
import matplotlib.pyplot as plt
from scipy.sparse.csgraph import shortest_path
import numpy as np
import pydot

In [2]:
# 1) One-hot data/identity matrix

n = 100
X = torch.eye(n)
# Calculate Euclidean distance matrix
dismat = torch.cdist(X, X)

delta_hyperbolicity(dismat)

0.0

In [3]:
# 2) Leaf node deltas

In [ ]:
n_nodes = 1000
max_children = 4

In [5]:
# Create tree

G = nx.Graph()

G.add_nodes_from(range(n_nodes))

# Start with node 0 as the root
remaining_nodes = set(range(1, n_nodes))
connected_nodes = {0}

while remaining_nodes:
    # Pick a random connected node to be a parent
    parent = np.random.choice(list(connected_nodes))
    
    # Determine number of children for this parent (1 to max_children)
    n_children = min(np.random.randint(1, max_children+1), len(remaining_nodes))
    
    # Randomly select children from remaining nodes
    children = np.random.choice(list(remaining_nodes), size=n_children, replace=False)
    
    # Add edges between parent and children
    for child in children:
        G.add_edge(parent, child)
        connected_nodes.add(child)
        remaining_nodes.remove(child)

leaf_nodes = [node for node, degree in G.degree() if degree == 1]
root_node = 0

adj_matrix = nx.to_numpy_array(G)

# Calculate shortest paths between all nodes
dist_matrix_full = shortest_path(adj_matrix, directed=False)

leaf_indices = np.array(leaf_nodes)
leaf_dismat = torch.from_numpy(dist_matrix_full[np.ix_(leaf_indices, leaf_indices)])

delta_hyperbolicity(leaf_dismat)

0.0

In [6]:
from manify.manifolds import ProductManifold
from manify.embedders.coordinate_learning import train_coords

In [9]:
CURVATURE = -1
DIMENSION = 2
N_SAMPLES = 100

pm = ProductManifold(signature=[(CURVATURE, DIMENSION)])
tree_emb = train_coords(pm,leaf_dismat,training_iterations=80)

  0%|          | 0/2200 [00:00<?, ?it/s]

ValueError: Loss is NaN